# Fase 3 - Dead reckoning de IMU (ver la deriva)

Entradas:  `results/MH_01_easy/fase1_sync.npz`, `results/MH_01_easy/fase2_calib.npz`
           y los CSV crudos de `mav0/imu0/` y `mav0/state_groundtruth_estimate0/`
Salidas:   `results/MH_01_easy/fase3_dr.npz`

**Objetivo:** integrar SOLO la IMU, medir la deriva, y con ello fijar de una vez
por todas los convenios de ejes, gravedad y bias.

**Criterio de éxito:** con estado inicial y bias tomados del GT, el error de
posición a los **5 s** debe ser **< 0.5 m**. Si es de metros, tienes un bug, no
"deriva normal".

Esta fase es el filtro de calidad más eficaz de todo el pipeline: casi todos los
errores de convenio se manifiestan aquí, y son *imposibles* de localizar más
tarde. En la Fase 5 solo verías "el filtro no converge" sin saber por qué.

**Por qué volvemos a los CSV crudos.** La Fase 1 guardó el GT muestreado en los
instantes de CÁMARA (20 Hz) y solo con posición y orientación. Aquí hace falta la
IMU a 200 Hz y las columnas `v_RS_R` (velocidad) y `b_w_RS_S` / `b_a_RS_S` (bias)
del ground truth, que solo están en el CSV.

**Aviso de frames.** La extrínseca `T_imu_cam` de la Fase 2 **no se usa** en esta
fase. El GT de EuRoC son poses del cuerpo `B`, y para `imu0` la extrínseca `T_BS`
es la identidad: `B = IMU`. Cámara e IMU no vuelven a mezclarse hasta la Fase 4.
Si te ves aplicando `T_imu_cam` aquí, párate.

In [1]:
from pathlib import Path

import matplotlib.pyplot as plt
import numpy as np
from scipy.spatial.transform import Rotation, Slerp

In [2]:
def encontrar_raiz(marcador="data"):
    actual = Path.cwd().resolve()
    for candidato in [actual, *actual.parents]:
        if (candidato / marcador).is_dir():
            return candidato
    raise FileNotFoundError(f"No encuentro '{marcador}/' hacia arriba desde {actual}")

In [3]:
RAIZ = encontrar_raiz()
ENTRADA_F1 = RAIZ / "results" / "MH_01_easy" / "fase1_sync.npz"
ENTRADA_F2 = RAIZ / "results" / "MH_01_easy" / "fase2_calib.npz"
SALIDA = RAIZ / "results" / "MH_01_easy" / "fase3_dr.npz"

np.set_printoptions(precision=8, suppress=True)

In [7]:
d1 = np.load(ENTRADA_F1, allow_pickle=True)
d2 = np.load(ENTRADA_F2, allow_pickle=True)

print("claves de fase1_sync.npz :", sorted(d1.files))
print("claves de fase2_calib.npz:", sorted(d2.files))

claves de fase1_sync.npz : ['K', 'T0_ns', 'T_imu_cam', 'accel', 'accel_medio_est', 'bias_giro_est', 'dist', 'g_body', 'gt_ba', 'gt_bw', 'gt_p', 'gt_p_cam', 'gt_q_cam_wxyz', 'gt_q_wxyz', 'gt_v', 'gt_v_cam', 'gyro', 'i0_estatico', 'i1_estatico', 'imu_fin', 'imu_ini', 'mask_gt', 'pitch0', 'residuo_gravedad', 'roll0', 'ruido_accelerometer_noise_density', 'ruido_accelerometer_random_walk', 'ruido_gyroscope_noise_density', 'ruido_gyroscope_random_walk', 'rutas_img', 'sigma_bias_giro', 't_arranque', 't_cam', 't_cam_gt', 't_despegue', 't_estatico', 't_gt', 't_imu', 't_motores']
claves de fase2_calib.npz: ['K', 'K_new', 'T_cam_imu', 'T_imu_cam', 'T_rel_gt', 'T_world_cam', 'ang_rel', 'desp_rel', 'dist', 'mapa_x', 'mapa_y', 'p_cam_world', 'roi', 't_cam_gt']


In [9]:
def sacar(npzs, *nombres, defecto=None):
    """Primera de 'nombres' que exista en ualquiera de los npz, o 'defectos'."""
    for npz in npzs:
        for nombre in nombres:
            if nombre in npz.files:
                return npz[nombre]
    if defecto is None:
        raise KeyError(f"ninguna de {nombres} está en los npz")
    print(f"  aviso: '{nombres[0]}' no está guardado, uso el valor por defecto {defecto}")
    return np.asarray(defecto)

In [10]:
rutas_img = sacar([d1], "rutas_img")
t_cam_gt = sacar([d1, d2], "t_cam_gt")

# Estos dos salieron de la Fase 1. 
T_ESTATICO = np.asarray(sacar([d1], "tramo_estatico", "t_estatico", defecto=[21.82, 30.98]), float)
T_ARRANQUE = float(sacar([d1], "t_arranque", defecto=45.82))

print(f"\ntramo estático = [{T_ESTATICO[0]:.2f}, {T_ESTATICO[1]:.2f}] s")
print(f"t_arranque     = {T_ARRANQUE:.2f} s")


tramo estático = [21.82, 30.98] s
t_arranque     = 45.82 s


## 3.0 - Datos crudos y base de tiempos

Dos cosas que conviene cerrar antes de integrar nada:

1. **Los timestamps de EuRoC son enteros de 64 bits en nanosegundos.** Leerlos
   como `float64` los cuantiza: 1.4e18 ns no cabe en una mantisa de 53 bits y la
   resolución efectiva pasa a ~256 ns. No es catastrófico (`dt` ≈ 5e6 ns), pero
   se arregla gratis restando el offset **en entero** y convirtiendo después. Es
   la última fila de la tabla de trampas de 3.7.

2. **La base de tiempos tiene que ser la misma que la de la Fase 1**, o
   `t_arranque` y el tramo estático apuntarán a otro sitio. Los nombres de
   fichero de `cam0` *son* el timestamp en ns, así que el offset que usaste en la
   Fase 1 se puede recuperar comparando con `t_cam_gt[0]` en vez de suponerlo.

In [15]:
# La ruta a mav0 se deduce de las imágenes de la Fase 1: .../mav0/cam0/data/<ts>.png
MAV0 = Path(str(rutas_img[0])).parents[2]
CSV_IMU = MAV0 / "imu0" / "data.csv"
CSV_GT = MAV0 / "state_groundtruth_estimate0" / "data.csv"

assert CSV_IMU.is_file(), f"no encuentro {CSV_IMU}"
assert CSV_GT.is_file(), f"no encuentro {CSV_GT}"
print(f"mav0 = {MAV0}")

mav0 = ..\data\MH_01_easy\mav0


In [16]:
def leer_csv_euroc(ruta, n_columnas):
    """
    CSV de EuRoC -> (timestamps en ns como int64, resto de columnas como float64).

    Dos pasadas a propósito: la primera columna se lee como ENTERO para no perder
    resolución temporal, el resto como flotante.
    """
    ts_ns = np.loadtxt(ruta, delimiter=",", skiprows=1, usecols=0, dtype=np.int64)
    datos = np.loadtxt(ruta, delimiter=",", skiprows=1, usecols=range(1, n_columnas), dtype=np.float64)
    return ts_ns, datos

In [18]:
ts_imu_ns, imu_datos = leer_csv_euroc(CSV_IMU, 7)
w_medida = imu_datos[:, 0:3]      # rad/s, frame body
a_medida = imu_datos[:, 3:6]      # m/s^2, frame body, FUERZA ESPECÍFICA

# state_groundtruth_estimate0: timestamp, p(3), q_wxyz(4), v(3), b_w(3), b_a(3)
ts_gt_ns, gt_datos = leer_csv_euroc(CSV_GT, 17)
gt_p = gt_datos[:, 0:3]
gt_q_wxyzgt_p = gt_datos[:, 0:3]
gt_q_wxyz = gt_datos[:, 3:7]
gt_v = gt_datos[:, 7:10]
gt_bg = gt_datos[:, 10:13]
gt_ba = gt_datos[:, 13:16]

print(f"IMU: {len(ts_imu_ns)} muestras")
print(f"GT : {len(ts_gt_ns)} muestras")

IMU: 36820 muestras
GT : 36382 muestras


In [19]:
# Base de tiempos: recupero el offset de la Fase 1 desde el nombre del primer PNG.
t_cam0_ns = int(Path(str(rutas_img[0])).stem)
OFFSET_NS = t_cam0_ns - int(round(float(t_cam_gt[0]) * 1e9))

t_imu = (ts_imu_ns - OFFSET_NS) / 1e9
t_gt = (ts_gt_ns - OFFSET_NS) / 1e9

print(f"offset de la Fase 1 = {OFFSET_NS} ns")
print(f"t_imu: [{t_imu[0]:8.3f}, {t_imu[-1]:8.3f}] s")
print(f"t_gt : [{t_gt[0]:8.3f}, {t_gt[-1]:8.3f}] s")
print(f"t_cam: [{t_cam_gt[0]:8.3f}, {t_cam_gt[-1]:8.3f}] s   <- de la Fase 1, debe solapar")

assert abs(t_imu[0] - float(t_cam_gt[0])) < 5.0, "las bases de tiempo no cuadran, revisa OFFSET_NS"

offset de la Fase 1 = 1403636578663555584 ns
t_imu: [   1.095,  185.190] s
t_gt : [   2.175,  184.080] s
t_cam: [   1.100,  182.950] s   <- de la Fase 1, debe solapar


In [20]:
# Estadística de dt: la IMU es de 200 Hz nominales, pero los dt NO son constantes.
dt_imu = np.diff(t_imu)
print(f"dt de la IMU:  media = {dt_imu.mean()*1e3:.4f} ms   std = {dt_imu.std()*1e6:.1f} us")
print(f"               min   = {dt_imu.min()*1e3:.4f} ms   max = {dt_imu.max()*1e3:.4f} ms")
print(f"frecuencia media = {1/dt_imu.mean():.2f} Hz")
assert dt_imu.min() > 0, "hay timestamps repetidos o desordenados en la IMU"

# Lo que habría costado leer el timestamp como float64 directamente:
dt_naive = np.diff(ts_imu_ns.astype(np.float64) / 1e9)
print(f"\ncuantización si el ts se lee como float64: max |Δdt| = "
      f"{np.abs(dt_naive - dt_imu).max()*1e9:.0f} ns")

dt de la IMU:  media = 5.0000 ms   std = 0.1 us
               min   = 4.9999 ms   max = 5.0002 ms
frecuencia media = 200.00 Hz

cuantización si el ts se lee como float64: max |Δdt| = 178 ns


## 3.1 - Las ecuaciones

Estado nominal: `x = (p_w, v_w, R_ws, b_g, b_a)`, donde `R_ws` lleva vectores del
frame body (`s`, el IMU) al mundo (`w`).

Medidas: `w_m` (rad/s, body), `a_m` (m/s², body, **fuerza específica**).

```
w = w_m - b_g - n_g
a = a_m - b_a - n_a

Ṙ_ws = R_ws · [w]×
v̇_w  = R_ws · a + g_w                 con g_w = (0, 0, -9.81)
ṗ_w  = v_w
ḃ_g  = n_bg          (random walk)
ḃ_a  = n_ba
```

**Por qué el acelerómetro no mide aceleración.** En caída libre marca cero; en
reposo marca `+g` en la dirección "arriba" del body. Lo que mide es la fuerza
específica `f = R_ws^T (a_w - g_w)`. Despejar `a_w = R_ws·f + g_w` es *la*
ecuación de la Fase 3, y el signo de `g_w` es donde se equivoca todo el mundo.

### Discretización

**Euler hacia adelante** (sencillo, error O(dt)):

```
R_{k+1} = R_k · Exp(w_k · dt)
a_w     = R_k · a_k + g_w
p_{k+1} = p_k + v_k·dt + 0.5·a_w·dt²
v_{k+1} = v_k + a_w·dt
```

**Punto medio / midpoint** (error O(dt²), el que hay que usar):

```
w̄       = 0.5 (w_k + w_{k+1})
R_{k+1} = R_k · Exp(w̄ · dt)
ā_w     = 0.5 (R_k·a_k + R_{k+1}·a_{k+1}) + g_w
p_{k+1} = p_k + v_k·dt + 0.5·ā_w·dt²
v_{k+1} = v_k + ā_w·dt
```

A 200 Hz la diferencia entre ambos en 5 s es de centímetros, pero se acumula. Y
midpoint es lo que usan VINS-Mono y la preintegración de GTSAM, así que es lo que
interesa tener interiorizado.

Ojo al orden `R · Exp(δ)`: la velocidad angular está medida **en el frame body**,
así que el incremento se compone **por la derecha**. `Exp(δ) · R` sería un
incremento expresado en el mundo, que no es lo que da el giróscopo. Este es
exactamente el mismo criterio que gobierna dónde va el "boxplus" en la
optimización sobre la variedad de la Fase 6.

In [21]:
def sombrero(v):
    """Matriz antisimétrica [v]x, tal que [v]x @ u = v × u."""
    v = np.asarray(v, dtype=np.float64)
    return np.array([[0.0, -v[2], v[1]],
                     [v[2], 0.0, -v[0]],
                     [-v[1], v[0], 0.0]])

In [22]:
def Exp(phi):
    """
    Exponencial de SO(3) (Rodrigues): vector de rotación (3,) -> matriz 3x3.

        Exp(phi) = I + (sin θ/θ)·[phi]x + ((1-cos θ)/θ²)·[phi]x²    con θ = |phi|

    Para θ pequeño esas divisiones pierden precisión, así que se usa el
    desarrollo I + [phi]x + [phi]x²/2. Con dt = 5 ms y 1 rad/s tenemos θ ~ 5e-3,
    lejos del umbral; pero θ = 0 exacto aparece en cuanto el dron está quieto.
    """
    phi = np.asarray(phi, dtype=np.float64)
    theta = np.linalg.norm(phi)
    S = sombrero(phi)
    if theta < 1e-7:
        return np.eye(3) + S + 0.5 * (S @ S)
    return (np.eye(3)
            + (np.sin(theta) / theta) * S
            + ((1.0 - np.cos(theta)) / (theta * theta)) * (S @ S))

In [24]:
def Log(R):
    """
    Logaritmo de SO(3): matriz 3x3 -> vector de rotación (3,), con |Log(R)| = ángulo.

    Cerca de θ = π el factor θ/(2 sen θ) explota, así que ahí se pasa por
    cuaternión. Es la métrica del error de actitud: |Log(R_gt^T · R_est)|.
    """
    coseno = np.clip((np.trace(R) - 1.0) / 2.0, -1.0, 1.0)
    theta = np.arccos(coseno)
    if theta > 3.0:
        return Rotation.from_matrix(R).as_rotvec()
    w = np.array([R[2, 1] - R[1, 2],
                    R[0, 2] - R[2, 0],
                    R[1, 0] - R[0, 1]])
    if theta < 1e-7:
        return 0.5 * w
    return (theta / (2.0 * np.sin(theta))) * w

In [25]:
# Tests de Exp/Log: ortonormalidad, det = +1, ida y vuelta y coherencia con scipy.
rng = np.random.default_rng(0)
err_rt, err_scipy = [], []
for _ in range(2000):
    u = rng.normal(size=3)
    phi = u / np.linalg.norm(u) * rng.uniform(0, np.pi - 1e-9)
    R = Exp(phi)
    assert np.allclose(R.T @ R, np.eye(3), atol=1e-12)
    assert abs(np.linalg.det(R) - 1.0) < 1e-12
    err_rt.append(np.linalg.norm(Log(R) - phi))
    err_scipy.append(np.abs(R - Rotation.from_rotvec(phi).as_matrix()).max())

print(f"|Log(Exp(phi)) - phi| : max = {max(err_rt):.2e} rad")
print(f"Exp vs scipy          : max = {max(err_scipy):.2e}")
print(f"ángulo de 1e-9 rad    : {np.abs(Exp([1e-9, 0, 0]) - Rotation.from_rotvec([1e-9, 0, 0]).as_matrix()).max():.2e}")
assert max(err_rt) < 1e-9 and max(err_scipy) < 1e-12

|Log(Exp(phi)) - phi| : max = 1.31e-13 rad
Exp vs scipy          : max = 8.88e-16
ángulo de 1e-9 rad    : 0.00e+00


In [26]:
G_W = np.array([0.0, 0.0, -9.81])   # gravedad en el frame mundo de EuRoC

## 3.2 - El GT como función continua del tiempo

Los timestamps del GT y los de la IMU **no coinciden**: son dos flujos distintos.
Para inicializar y para comparar hace falta evaluar el GT en instantes arbitrarios.

Posición, velocidad y bias se interpolan linealmente. La orientación no: un
cuaternión interpolado componente a componente **deja de ser unitario**, y la
rotación resultante no es la del camino más corto. Lo correcto es **SLERP**, que
interpola a lo largo de la geodésica de la esfera.

A 200 Hz y con movimiento suave la diferencia entre lerp normalizado y slerp es
de ~1e-6 rad, o sea irrelevante aquí. El hábito importa porque en la Fase 7, al
alinear trayectorias con muestreos distintos y saltos mayores, sí importa.

In [27]:
# Interpoladores del GT: se construyen una vez y se reutilizan.
_slerp_gt = Slerp(t_gt, Rotation.from_quat(gt_q_wxyz[:, [1, 2, 3, 0]]))   # wxyz -> xyzw

def gt_en(t):
    """
    Evalúa el GT en los instantes `t` (escalar o array).
    Devuelve (p, R, v, bg, ba) con formas (N,3), (N,3,3), (N,3), (N,3), (N,3).
    """
    t = np.atleast_1d(np.asarray(t, dtype=np.float64))
    assert t.min() >= t_gt[0] - 1e-9 and t.max() <= t_gt[-1] + 1e-9, \
        f"t fuera del rango del GT [{t_gt[0]:.3f}, {t_gt[-1]:.3f}] s"
    t = np.clip(t, t_gt[0], t_gt[-1])

    def interp(M):
        return np.column_stack([np.interp(t, t_gt, M[:, j]) for j in range(3)])

    return interp(gt_p), _slerp_gt(t).as_matrix(), interp(gt_v), interp(gt_bg), interp(gt_ba)

In [28]:
# COMPROBACIÓN CLAVE: el mundo de EuRoC está alineado con la gravedad.
# En el tramo estático, R_ws · a_medida debe dar (0, 0, +9.81): en reposo el
# acelerómetro mide la reacción normal, que apunta hacia ARRIBA en el mundo.
m = (t_imu >= T_ESTATICO[0]) & (t_imu <= T_ESTATICO[1])
_, R_estatico, _, _, _ = gt_en(t_imu[m])
a_mundo = np.einsum("kij,kj->ki", R_estatico, a_medida[m]).mean(axis=0)

print(f"media de R_ws · a_medida en el tramo estático: {a_mundo}")
print(f"  módulo = {np.linalg.norm(a_mundo):.4f} m/s^2")
print(f"  desviación respecto de la vertical = "
      f"{np.degrees(np.arctan2(np.linalg.norm(a_mundo[:2]), a_mundo[2])):.3f}°")

assert a_mundo[2] > 9.0, "el eje z del mundo no apunta hacia arriba, o el cuaternión está mal convertido"

media de R_ws · a_medida en el tramo estático: [-0.03728378  0.14030779  9.77481355]
  módulo = 9.7759 m/s^2
  desviación respecto de la vertical = 0.851°


Si esa media sale ≈ `(0, 0, +9.81)` con menos de ~1° de desviación quedan
confirmadas tres cosas de golpe: el eje z del mundo apunta hacia arriba, el
acelerómetro mide fuerza específica (no aceleración), y el cuaternión del GT está
bien convertido (`wxyz` → `xyzw` y no al revés).

El módulo no será exactamente 9.81: la diferencia es bias del acelerómetro más
gravedad local. Volvemos a ella en el Experimento D.

## 3.3 - El integrador

In [ ]:
def integrar_imu(t, w_m, a_m, p0, v0, R0, bg, ba,
                 metodo="midpoint", g_w=G_W, orden="derecha", dt_fijo=None):
    """
    Dead reckoning de IMU. Devuelve (p, v, R) de formas (M,3), (M,3), (M,3,3).

    t          : (M,) instantes en segundos
    w_m, a_m   : (M,3) medidas crudas de giróscopo y acelerómetro
    p0, v0, R0 : estado inicial en el mundo
    bg, ba     : bias, (3,) constante o (M,3) variable en el tiempo (broadcasting)
    metodo     : "euler" | "midpoint"

    Los tres últimos argumentos existen SOLO para los tests negativos de la sección de trampas.
    En uso normal no se tocan.
    """
    M = len(t)
    w = w_m - bg
    a = a_m - ba
    
    p = np.zeros((M, 3))
    v = np.zeros((M, 3))
    R = np.zeros((M, 3, 3))
    p[0], v[0], R[0] = p0, v0, R0
    
    for k in range(M - 1):
        dt = dt_fijo if dt_fijo is not None else t[k+1] - t[k]
        if dt <= 0:
            p[k + 1], v[k + 1], R[k + 1] = p[k], v[k], R[k]
            continue
            
        if metodo == "euler":
            incremento = Exp(w[k] * dt)
            R_sig = R[k] @ incremento if orden == "derecha" else incremento @ R[k]
            a_w = R[k] @ a[k] + g_w
        else:
            w_barra = 0.5 * (w[k] + w[k + 1])
            incremento = Exp(w_barra * dt)
            R_sig = R[k] @ incremento if orden == "derecha" else incremento @ R[k]
            a_w = 0.5 * (R[k] @ a[k] + R_sig @ a[k + 1]) + g_w
            
        p[k + 1] = p[k] + v[k] * dt + 0.5 * a_w * dt ** 2
        v[k + 1] = v[k] + a_w * dt
        R[k + 1] = R_sig     # se escribe AL FINAL: arriba R[k] tiene que seguir siendo el viejo

    return p, v, R
        